# Tutorial: additive and zero-sum sequences

`zero_sum_sequences` represents finite multisets over additive parents and computes zero-sum atoms, factorizations, and automorphism orbits.  The examples below use $C_3$ and $C_2\oplus C_4$ and run with ordinary Python or on Binder.

## 1. Configure a sequence space

An `AdditiveSequenceSpace` stores an additive parent and an upper bound for its Davenport constant.  Calling the space constructs immutable finite multisets over that parent.  A bound below the actual Davenport constant can omit atoms and make factorization results incomplete.

For $C_3=\mathbb Z/3\mathbb Z$, the Davenport constant is 3.

In [ ]:
from zero_sum_sequences import (
    AdditiveSequenceSpace,
    AtomCatalogue,
    AutomorphismAction,
    FactorizationSolver,
    FiniteAdditiveGroup,
)

modulo_three = FiniteAdditiveGroup(
    range(3),
    zero=0,
    add=lambda left, right: (left + right) % 3,
    coerce=lambda value: int(value) % 3,
)
C3 = AdditiveSequenceSpace(modulo_three, davenport_bound=3)
print('base parent:', C3.base_parent)
print('Davenport bound:', C3.davenport_bound)

## 2. Construct and inspect sequences

Terms are coerced through the configured parent and stored as a canonical multiplicity table, so their input order is irrelevant.  A nonempty zero-sum sequence is an *atom* if it has no proper nonempty zero-sum subsequence.  `map_terms(mapping)` applies a map term by term.

In [ ]:
sample = C3([2, 1, 2, 1])
negated_sample = sample.map_terms(lambda term: -term)

print('sequence:', sample)
print('parent is C3:', sample.parent() is C3)
print('terms:', tuple(sample))
print('length:', len(sample))
print('support:', sample.support)
print('multiplicities:', sample.multiplicities)
print('total:', sample.total())
print('zero-sum:', sample.is_zero_sum())
print('atom:', sample.is_atom())
print('termwise negation:', negated_sample)
print('negation fixes sequence:', negated_sample == sample)

## 3. Multiset arithmetic

Addition combines multiplicities, subtraction removes a subsequence, and multiplication by a nonnegative integer repeats a sequence.  `divides` tests the subsequence relation.  All results remain in the same sequence space.

In [ ]:
pair = C3([1, 2])
triple_pair = 3 * pair

print('pair:', pair)
print('pair + pair:', pair + pair)
print('sample - pair:', sample - pair)
print('3 * pair:', triple_pair)
print('pair divides sample:', pair.divides(sample))
print('triple pair divides sample:', triple_pair.divides(sample))

## 4. Factorizations and length sets

Consider $S=1^3 2^3$ over $C_3$.  It has two visible factorizations into atoms:

$$S=(1^3)(2^3)=(1\cdot2)^3.$$

Their lengths are 2 and 3.  `length_set()` computes the complete set, while `factorization_witnesses()` keeps one deterministic witness for each length.

In [ ]:
sequence = C3([1, 1, 1, 2, 2, 2])
one_cubed = C3([1, 1, 1])
two_cubed = C3([2, 2, 2])
mixed_pair = C3([1, 2])

lengths = sequence.length_set()
witnesses = sequence.factorization_witnesses()

def format_factorization(factors):
    return '  *  '.join(str(factor) for factor in factors) or '1'

print('length set:', sorted(lengths))
for length, factors in witnesses.items():
    print(f'length {length}: {format_factorization(factors)}')

`factorizations()` enumerates every unordered factorization exactly once.  For larger examples, prefer the length set or one witness per length.

In [ ]:
all_factorizations = sorted(sequence.factorizations(), key=len)
for factors in all_factorizations:
    print(f'length {len(factors)}: {format_factorization(factors)}')

## 5. Reuse a factorization solver

When several queries use the same input, `FactorizationSolver` reuses its discovered atoms and remainder DAG.  Its statistics report the numbers of atom divisors, remainder states, and transitions.

In [ ]:
solver = FactorizationSolver(sequence)
solver_lengths = solver.length_set()
solver_witnesses = solver.factorization_witnesses()
stats = solver.statistics

print('length set:', sorted(solver_lengths))
print('witness lengths:', sorted(solver_witnesses))
print('candidate atoms:', stats.candidate_atoms)
print('remainder states:', stats.states)
print('transitions:', stats.transitions)

## 6. Inspect the factorization DAG

`sequence.factorization_digraph()` returns the remainder DAG.  Each vertex is a remainder sequence, and each edge records the removed atom.  A path from the input to the empty sequence encodes a factorization.

In [ ]:
import networkx as nx

dag = sequence.factorization_digraph()
labeled_edges = tuple(dag.edges(data='atom'))
edges = sorted(
    (str(source), str(target), str(atom))
    for source, target, atom in labeled_edges
)

print(f'vertices: {dag.number_of_nodes()}; edges: {dag.number_of_edges()}')
for source, target, atom in edges:
    print(f'{source}  -- remove {atom} -->  {target}')

path_lengths = {
    len(path) - 1 for path in nx.all_simple_paths(dag, sequence, C3())
}
print('acyclic:', nx.is_directed_acyclic_graph(dag))
print('path lengths:', sorted(path_lengths))

In [ ]:
import matplotlib.pyplot as plt

positions = {}
for depth, nodes in enumerate(nx.topological_generations(dag)):
    ordered_nodes = sorted(nodes, key=str)
    offset = (len(ordered_nodes) - 1) / 2
    for index, node in enumerate(ordered_nodes):
        positions[node] = (index - offset, -depth)

node_labels = {node: str(node) for node in dag}
edge_labels = {
    (source, target): str(atom)
    for source, target, atom in dag.edges(data='atom')
}

figure, axis = plt.subplots()
nx.draw_networkx_edges(
    dag,
    pos=positions,
    node_size=500,
    node_shape='s',
    arrowsize=18,
    ax=axis,
)
nx.draw_networkx_labels(
    dag,
    pos=positions,
    labels=node_labels,
    font_size=10,
    bbox={
        'boxstyle': 'round,pad=0.45',
        'facecolor': '#e8eef7',
        'edgecolor': '#607089',
    },
    ax=axis,
)
nx.draw_networkx_edge_labels(
    dag,
    pos=positions,
    edge_labels=edge_labels,
    font_size=10,
    rotate=False,
    ax=axis,
)
axis.set_axis_off()
figure.tight_layout()

## 7. Enumerate reduced atoms

For a finite group $G$, every atom has length at most its Davenport constant $D(G)$.  `enumerate_atom_catalogue()` enumerates the reduced atoms through the configured bound, omitting the identity singleton.

For $C_2\oplus C_4$, the Davenport constant is $1+(2-1)+(4-1)=5$, and the complete enumeration is small.

In [ ]:
from collections import Counter

group = FiniteAdditiveGroup.cyclic_product(2, 4)
C2xC4 = AdditiveSequenceSpace(group, davenport_bound=5)
c2_c4_catalogue = C2xC4.enumerate_atom_catalogue()
atoms_by_length = Counter(map(len, c2_c4_catalogue))

print('total reduced atoms:', len(c2_c4_catalogue))
for length, count in sorted(atoms_by_length.items()):
    print(f'length {length}: {count}')
print(
    'all entries are reduced atoms:',
    all(atom.is_atom() and group.zero() not in atom for atom in c2_c4_catalogue),
)

## 8. Automorphism orbits

Automorphisms act term by term; canonicalization handles the order of sequence terms.  `FiniteAdditiveGroup.cyclic_product(...)` supplies generators for the full automorphism group.  The induced action is cached on first use.

The eight atoms of length 5 over $C_2\oplus C_4$ form one orbit.  An orbit witness gives a shortest generator word between two atoms and can display the images of the standard additive generators.

In [ ]:
maximal_atom = C2xC4([(0, 1)] * 3 + [(1, 0), (1, 1)])
other_maximal_atom = C2xC4([(0, 1), (1, 0)] + [(1, 1)] * 3)
maximal_orbit = maximal_atom.orbit()
orbit_witness = maximal_atom.orbit_witness(other_maximal_atom)

print('orbit size:', len(maximal_orbit))
print('same orbit:', maximal_atom.is_in_same_orbit(other_maximal_atom))
print('generator word:', orbit_witness.generator_indices)
orbit_witness.show()

action = AutomorphismAction(group.automorphism_generators())
length_five_atoms = {atom for atom in c2_c4_catalogue if len(atom) == 5}
print('orbit contains every length-5 atom:', set(maximal_orbit) == length_five_atoms)
print(
    'witness maps atom to target:',
    action.apply_word(maximal_atom, orbit_witness) == other_maximal_atom,
)

## 9. Reuse a complete atom catalogue

An `AtomCatalogue` indexes precomputed atoms for reuse.  For the earlier $C_3$ input, the relevant atoms are $1^3$, $2^3$, and $1\cdot2$.  Complete factorization results require every atom divisor of the sequence.

In [ ]:
relevant_atoms = (one_cubed, two_cubed, mixed_pair)
catalogue = AtomCatalogue(C3, relevant_atoms)
catalogue_divisors = tuple(catalogue.divisors(sequence))

print('catalogue atoms:', [str(atom) for atom in catalogue])
print('divisors of S:', [str(atom) for atom in catalogue_divisors])
print(
    'length set with catalogue:',
    sorted(sequence.length_set(atom_catalogue=catalogue)),
)
catalogue_solver = FactorizationSolver(sequence, atom_catalogue=catalogue)
print('solver statistics:', catalogue_solver.statistics)